In [ ]:
PROPOSITION_ID = "C1"
STANCE_COLUMN = "C1"
RELEVANCE_COLUMN = "I1"
SENTIMENT_COLUMN = "S1"

CLASSIFICATION_RUN_ID = f"d1_{PROPOSITION_ID.lower()}_stance"

COMMENTS_PATH = "data/silver/experiments/d1/d1.csv"
LABELS_PATH = "/Users/lau/Downloads/comments_class_cleaned.csv"

OUTPUT_DIR = f"outputs/stances/{CLASSIFICATION_RUN_ID}"

PROPOSITION = "¿Cuál es su nivel de apoyo hacia el candidato presidencial Iván Cepeda?"
CONTEXT = (
    "Las elecciones presidenciales de Colombia de 2026 se desarrollan en un contexto de alta "
    "polarización política. El debate público ha estado marcado por discusiones en torno a "
    "propuestas del gobierno actual, liderado por Gustavo Petro, incluyendo iniciativas promovidas "
    "en las distintas reformas propuestas por el gobierno. En este escenario, distintos sectores "
    "políticos —de izquierda, centro y derecha— presentan candidatos con trayectorias diversas, "
    "mientras que temas como la economía, la seguridad, las reformas sociales y la institucionalidad "
    "ocupan un lugar central en la agenda nacional."
)
TOPIC_ID = "c1_ivan_cepeda"
CONTEXT_VERSION = "1"
OVERWRITE_OUTPUTS = True

# Build precomputed `CommentStance` artifacts

Adaptación experimental y reproducible de etiquetas existentes al contrato real de `CommentStance`. El CSV limpio conserva la identidad autoritativa del comment stream; el CSV etiquetado se registra como fuente de etiquetas separada. No se realizan llamadas a OpenAI ni a otros proveedores.

## 1. Preparación y funciones de auditoría

Para C2–C4 se reutiliza el mismo notebook cambiando los parámetros de proposición y las columnas `C*`, `I*`, `S*`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
try:
    from IPython.display import display
except ImportError:  # permite validación headless con el entorno del proyecto
    display = print

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "youtube_pipeline").is_dir():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next(path for path in candidates if (path / "youtube_pipeline").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from youtube_pipeline.comment_stance import (
    CLASSIFIED, NOT_APPLICABLE, UNCERTAIN, CommentStance,
)
from youtube_pipeline.experimental_input import (
    CAUSAL_MINIMAL_POLICY_ID, ExperimentalInputPolicyConfig,
    prepare_causal_minimal_batch, source_sha256,
)
from youtube_pipeline.signal_observations import to_json_mapping
from youtube_pipeline.storage import canonicalize_comment_event_time

COMMENTS_FILE = (REPO_ROOT / COMMENTS_PATH).resolve()
LABELS_FILE = Path(LABELS_PATH).expanduser().resolve()
OUTPUT_ROOT = (REPO_ROOT / OUTPUT_DIR).resolve()

def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()

def audit_input(frame: pd.DataFrame, source: str) -> dict[str, object]:
    duplicate_mask = frame["comment_id"].duplicated(keep=False) if "comment_id" in frame else pd.Series(False, index=frame.index)
    return {
        "source": source,
        "rows": len(frame),
        "unique_comment_id": frame["comment_id"].nunique(dropna=True),
        "duplicate_comment_ids": frame.loc[duplicate_mask, "comment_id"].nunique(dropna=True),
        "duplicate_extra_rows": int(frame["comment_id"].duplicated().sum()),
        "null_comment_id": int(frame["comment_id"].isna().sum()),
        "columns": list(frame.columns),
    }

def deduplicate_exact(frame: pd.DataFrame, source: str):
    if frame["comment_id"].isna().any():
        raise ValueError(f"{source}: null comment_id cannot be exported")
    duplicate_ids = frame.loc[frame["comment_id"].duplicated(keep=False), "comment_id"].drop_duplicates().tolist()
    conflicts = [
        comment_id for comment_id in duplicate_ids
        if len(frame.loc[frame["comment_id"] == comment_id].drop_duplicates()) != 1
    ]
    if conflicts:
        raise ValueError(f"{source}: conflicting duplicate comment_id values: {conflicts[:20]}")
    return frame.drop_duplicates(subset=["comment_id"], keep="first").copy(), duplicate_ids, conflicts

def normalize_nullable_integer(series: pd.Series, allowed: set[int], field: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    malformed = series.notna() & numeric.isna()
    non_integral = numeric.notna() & numeric.mod(1).ne(0)
    outside = numeric.notna() & ~numeric.isin(allowed)
    invalid = malformed | non_integral | outside
    if invalid.any():
        examples = series.loc[invalid].value_counts(dropna=False).head(20).to_dict()
        raise ValueError(f"{field}: malformed/out-of-range values: {examples}")
    return numeric.astype("Int64")

def normalize_boolean(series: pd.Series, field: str) -> pd.Series:
    mapping = {True: True, False: False, 1: True, 0: False, "true": True, "false": False, "1": True, "0": False}
    def one(value):
        if pd.isna(value):
            return pd.NA
        key = value.strip().lower() if isinstance(value, str) else value
        return mapping.get(key, pd.NA)
    result = series.map(one).astype("boolean")
    invalid = series.notna() & result.isna()
    if invalid.any():
        raise ValueError(f"{field}: non-interpretable values: {series.loc[invalid].value_counts(dropna=False).to_dict()}")
    if result.isna().any():
        raise ValueError(f"{field}: null relevance cannot determine an empty stance")
    return result

## 2. Auditoría de inputs y deduplicación exacta

In [ ]:
comments_raw = pd.read_csv(COMMENTS_FILE)
labels_raw = pd.read_csv(LABELS_FILE)
required_comment_columns = {"video_id", "comment_id", "text", "author_id", "published_at"}
required_label_columns = {"comment_id", "text", STANCE_COLUMN, RELEVANCE_COLUMN, SENTIMENT_COLUMN}
if missing := required_comment_columns - set(comments_raw.columns):
    raise ValueError(f"Comments source lacks columns: {sorted(missing)}")
if missing := required_label_columns - set(labels_raw.columns):
    raise ValueError(f"Label source lacks columns: {sorted(missing)}")

input_audit = pd.DataFrame([
    audit_input(comments_raw, COMMENTS_PATH),
    audit_input(labels_raw, LABELS_PATH),
])
display(input_audit)

for name, row in input_audit.set_index("source").iterrows():
    if row["rows"] != 78_045 or row["unique_comment_id"] != 78_016:
        print(f"AUDIT WARNING: {name} differs from expected D1 counts")

comments_unique, comment_duplicate_ids, comment_conflicts = deduplicate_exact(comments_raw, "comments")
labels_unique, label_duplicate_ids, label_conflicts = deduplicate_exact(labels_raw, "labels")
duplicate_summary = pd.DataFrame([
    {"source": "comments", "duplicate_ids": len(comment_duplicate_ids), "conflicts": len(comment_conflicts), "unique_rows": len(comments_unique)},
    {"source": "labels", "duplicate_ids": len(label_duplicate_ids), "conflicts": len(label_conflicts), "unique_rows": len(labels_unique)},
])
display(duplicate_summary)

## 3. Join exclusivo por `comment_id`

In [ ]:
join_audit = comments_unique[["comment_id"]].merge(
    labels_unique[["comment_id"]], on="comment_id", how="outer", indicator=True, validate="one_to_one"
)
join_counts = join_audit["_merge"].value_counts().to_dict()
matched = int(join_counts.get("both", 0))
unmatched_comments = int(join_counts.get("left_only", 0))
unmatched_labels = int(join_counts.get("right_only", 0))
coverage = matched / len(comments_unique) if len(comments_unique) else 0.0
join_summary = pd.DataFrame([{
    "matched": matched, "unmatched_comments": unmatched_comments,
    "unmatched_labels": unmatched_labels, "coverage": coverage,
}])
display(join_summary)
if unmatched_comments or unmatched_labels:
    raise ValueError("Join coverage is incomplete; export is blocked")

## 4. Auditoría de diferencias de texto (no participa en el join)

In [ ]:
text_audit = comments_unique[["comment_id", "text"]].merge(
    labels_unique[["comment_id", "text"]], on="comment_id", suffixes=("_comment", "_label"), validate="one_to_one"
)
def text_value(value):
    return "" if pd.isna(value) else str(value)
def normalize_line_breaks(value):
    return text_value(value).replace("\r\n", "\n").replace("\r", "\n")
def text_difference_kind(row):
    left, right = text_value(row.text_comment), text_value(row.text_label)
    if left == right:
        return "identical"
    if normalize_line_breaks(left) == normalize_line_breaks(right):
        return "line-break normalization"
    if normalize_line_breaks(left).strip() == normalize_line_breaks(right).strip():
        return "leading/trailing whitespace"
    return "other transformations"
text_audit["difference_kind"] = text_audit.apply(text_difference_kind, axis=1)
text_difference_summary = text_audit["difference_kind"].value_counts().rename_axis("kind").reset_index(name="count")
display(text_difference_summary)
nontrivial_text_differences = text_audit.loc[
    text_audit["difference_kind"] == "other transformations",
    ["comment_id", "text_comment", "text_label", "difference_kind"],
].copy()
error_like_values = text_audit.loc[
    text_audit["text_comment"].fillna("").astype(str).str.contains("#ERROR!", regex=False)
    | text_audit["text_label"].fillna("").astype(str).str.contains("#ERROR!", regex=False),
    ["comment_id", "text_comment", "text_label"],
]
display(nontrivial_text_differences)
display(error_like_values)

## 5. Normalización C1 / I1 / S1 y mapeo

`S1` se audita como sentimiento 1–3, pero nunca se utiliza para construir la postura.

In [ ]:
labels_work = labels_unique.copy()
labels_work["stance_normalized"] = normalize_nullable_integer(labels_work[STANCE_COLUMN], {1, 2, 3, 4, 5}, STANCE_COLUMN)
labels_work["relevance_normalized"] = normalize_boolean(labels_work[RELEVANCE_COLUMN], RELEVANCE_COLUMN)
labels_work["sentiment_normalized"] = normalize_nullable_integer(labels_work[SENTIMENT_COLUMN], {1, 2, 3}, SENTIMENT_COLUMN)

def map_status(row):
    if pd.notna(row.stance_normalized):
        return CLASSIFIED
    return NOT_APPLICABLE if not bool(row.relevance_normalized) else UNCERTAIN
labels_work["classification_status"] = labels_work.apply(map_status, axis=1)

display(labels_work[STANCE_COLUMN].value_counts(dropna=False).sort_index().rename("count").to_frame())
display(labels_work[RELEVANCE_COLUMN].value_counts(dropna=False).rename("count").to_frame())
display(labels_work[SENTIMENT_COLUMN].value_counts(dropna=False).sort_index().rename("count").to_frame())
status_counts = labels_work["classification_status"].value_counts().to_dict()
status_summary = pd.DataFrame([{
    CLASSIFIED: int(status_counts.get(CLASSIFIED, 0)),
    NOT_APPLICABLE: int(status_counts.get(NOT_APPLICABLE, 0)),
    UNCERTAIN: int(status_counts.get(UNCERTAIN, 0)),
    "TOTAL": len(labels_work),
}])
display(status_summary)
if PROPOSITION_ID == "C1":
    expected = {CLASSIFIED: 45_289, NOT_APPLICABLE: 32_714, UNCERTAIN: 13}
    actual = {key: int(status_counts.get(key, 0)) for key in expected}
    if actual != expected or sum(actual.values()) != len(labels_work):
        raise ValueError(f"Unexpected C1 status counts; expected={expected}, actual={actual}")
display(labels_work.groupby("classification_status", sort=True).head(3)[
    ["comment_id", STANCE_COLUMN, RELEVANCE_COLUMN, SENTIMENT_COLUMN, "classification_status"]
].sort_values(["classification_status", "comment_id"]))

## 6. Construcción usando el contrato real `CommentStance`

In [ ]:
canonical_comments = canonicalize_comment_event_time(comments_raw)
prepared = prepare_causal_minimal_batch(
    canonical_comments, ExperimentalInputPolicyConfig(), source_path=COMMENTS_FILE,
)
record_by_id = {record.comment_id: record for record in prepared.records}
if set(record_by_id) != set(labels_work["comment_id"]):
    raise ValueError("Label IDs differ from the causal_minimal_v1 comment universe")
label_by_id = labels_work.set_index("comment_id")
stances = []
for record in prepared.records:
    label = label_by_id.loc[record.comment_id]
    status = str(label["classification_status"])
    category = int(label["stance_normalized"]) if status == CLASSIFIED else None
    stances.append(CommentStance(
        comment_id=record.comment_id,
        author_id=record.author_id,
        event_time_utc=record.event_time_utc,
        proposition_id=PROPOSITION_ID,
        stance_category=category,
        classification_status=status,
        quality="passed",
    ))
stances = tuple(sorted(stances, key=lambda item: (item.event_time_utc, item.comment_id)))
if len(stances) != 78_016 or len({item.comment_id for item in stances}) != len(stances):
    raise ValueError("CommentStance cardinality/uniqueness check failed")
stance_rows = [to_json_mapping(item) for item in stances]
pd.DataFrame(stance_rows).head()

## 7. Exportación determinista y provenance precomputado

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
stance_path = OUTPUT_ROOT / "comment_stances.jsonl"
manifest_path = OUTPUT_ROOT / "classifier_manifest.json"
resolved_config_path = OUTPUT_ROOT / "resolved_config.json"
target_paths = (stance_path, manifest_path, resolved_config_path)
if not OVERWRITE_OUTPUTS and any(path.exists() for path in target_paths):
    raise FileExistsError("Output exists; set OVERWRITE_OUTPUTS=True explicitly")

jsonl_payload = "".join(
    json.dumps(row, ensure_ascii=False, sort_keys=True, allow_nan=False) + "\n"
    for row in stance_rows
)
stance_path.write_text(jsonl_payload, encoding="utf-8")
comment_ids_hash = hashlib.sha256(
    "\n".join(sorted(item.comment_id for item in stances)).encode("utf-8")
).hexdigest()
comment_source_hash = source_sha256(COMMENTS_FILE)
label_source_hash = source_sha256(LABELS_FILE)
context_hash = hashlib.sha256(CONTEXT.encode("utf-8")).hexdigest()
proposition_snapshot = {
    "proposition_id": PROPOSITION_ID, "proposition_text": PROPOSITION,
    "context": CONTEXT, "context_version": CONTEXT_VERSION,
}
mapping_policy = {
    "policy_id": "precomputed_labels_c_i_v1",
    "classified_rule": f"{STANCE_COLUMN} in {{1,2,3,4,5}}",
    "not_applicable_rule": f"{STANCE_COLUMN} is null and {RELEVANCE_COLUMN} is false",
    "uncertain_rule": f"{STANCE_COLUMN} is null and {RELEVANCE_COLUMN} is true",
    "sentiment_used_for_stance": False,
}
manifest = {
    "artifact_version": "precomputed_comment_stance_v1",
    "execution_status": "imported_precomputed_labels",
    "classifier_run_id": CLASSIFICATION_RUN_ID,
    "stance_source_type": "PRECOMPUTED_LABELS",
    "provider": None, "model": None, "model_calls": 0,
    "prompt_id": None, "prompt_version": None,
    "topic_id": TOPIC_ID,
    "proposition_id": PROPOSITION_ID,
    "proposition": proposition_snapshot,
    "context_hash": context_hash,
    "label_field": STANCE_COLUMN,
    "relevance_field": RELEVANCE_COLUMN,
    "sentiment_field": SENTIMENT_COLUMN,
    "sentiment_used_for_stance": False,
    "classification_policy": mapping_policy,
    "comment_source": COMMENTS_PATH,
    "comment_source_path": COMMENTS_PATH,
    "comment_source_sha256": comment_source_hash,
    "label_source": str(LABELS_FILE),
    "label_source_path": str(LABELS_FILE),
    "label_source_sha256": label_source_hash,
    "input_ref": str(COMMENTS_FILE),
    "input_source_sha256": comment_source_hash,
    "input_policy_id": CAUSAL_MINIMAL_POLICY_ID,
    "source_rows": len(comments_raw),
    "label_source_rows": len(labels_raw),
    "prepared_unique_comments": len(prepared.records),
    "matched_comment_count": matched,
    "classified_count": int(status_counts.get(CLASSIFIED, 0)),
    "not_applicable_count": int(status_counts.get(NOT_APPLICABLE, 0)),
    "uncertain_count": int(status_counts.get(UNCERTAIN, 0)),
    "counts_by_status": {key: int(value) for key, value in sorted(status_counts.items())},
    "duplicate_comment_ids": len(comment_duplicate_ids),
    "duplicate_comment_ids_by_source": {"comments": len(comment_duplicate_ids), "labels": len(label_duplicate_ids)},
    "duplicate_conflict_count": len(comment_conflicts) + len(label_conflicts),
    "classified_comment_ids_sha256": comment_ids_hash,
    "comment_ids_hash": comment_ids_hash,
    "artifact_path": str(stance_path),
    "artifact_sha256": sha256_bytes(jsonl_payload.encode("utf-8")),
    "ordering": "event_time_utc_then_comment_id",
    "openai_calls": 0, "serper_calls": 0,
}
resolved_config = {
    "run_id": CLASSIFICATION_RUN_ID,
    "classification_run_id": CLASSIFICATION_RUN_ID,
    "proposition_id": PROPOSITION_ID,
    "proposition": PROPOSITION,
    "context": CONTEXT,
    "context_version": CONTEXT_VERSION,
    "topic_id": TOPIC_ID,
    "stance_mode": "precomputed_labels_v1",
    "stance_source_type": "PRECOMPUTED_LABELS",
    "comment_source": COMMENTS_PATH,
    "comment_source_sha256": comment_source_hash,
    "label_source": str(LABELS_FILE),
    "label_source_sha256": label_source_hash,
    "stance_column": STANCE_COLUMN,
    "relevance_column": RELEVANCE_COLUMN,
    "sentiment_column": SENTIMENT_COLUMN,
    "sentiment_used_for_stance": False,
    "mapping_policy": mapping_policy,
    "model": None, "provider": None, "model_calls": 0,
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
resolved_config_path.write_text(json.dumps(resolved_config, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")

## 8. Resumen de artefactos

In [ ]:
artifact_summary = pd.DataFrame([
    {"artifact": "comment_stances", "path": str(stance_path), "records": len(stances), "sha256": manifest["artifact_sha256"]},
    {"artifact": "classifier_manifest", "path": str(manifest_path), "records": 1, "sha256": source_sha256(manifest_path)},
    {"artifact": "resolved_config", "path": str(resolved_config_path), "records": 1, "sha256": source_sha256(resolved_config_path)},
])
display(artifact_summary)
print(json.dumps({
    "status": "exported",
    "stance_source_type": "PRECOMPUTED_LABELS",
    "records": len(stances),
    "counts_by_status": manifest["counts_by_status"],
    "openai_calls": 0,
    "serper_calls": 0,
}, ensure_ascii=False, indent=2))